# 04 — Phát hiện ngoại lai

**Mục tiêu:** so sánh 5 phương pháp phát hiện ngoại lai và quyết định cách xử lý.

**Rubric:** Bảng — phát hiện ngoại lai và mã hóa (10%, cùng notebook 05).

**Đề yêu cầu:**
- [ ] IQR và Z-score.
- [ ] Isolation Forest với contamination ∈ {0.01, 0.05, 0.1}.
- [ ] LOF với n_neighbors ∈ {10, 20, 50}.
- [ ] DBSCAN.
- [ ] Báo cáo tỷ lệ phát hiện, độ chồng lấp giữa các tập (Jaccard) và tác động lên phân phối sau khi loại bỏ bằng kiểm định KS.

**Đầu vào:** development set đã điền thiếu theo lựa chọn ở notebook 03.

**Đầu ra:** `results/tabular/tables/outlier_rates.csv`, ma trận Jaccard, bảng KS; quyết định xóa/cắt ngưỡng/giữ.
---
*Quy ước khung:* ô **[Code]** sẽ được thay bằng ô code; ô **[Nhận xét]** chỉ viết sau khi có kết quả thật. Mọi score/p-value phải được **in ra và gán vào biến Python** (yêu cầu của đề).

## 0. Thiết lập

> **[Code]** Import thư viện; tìm gốc repo (`find_project_root`, nên chuyển vào `src/data_utils.py`); đọc `data/raw/home_credit/application_train.csv`; nạp `data/splits/home_credit_split_ids.csv` và **chỉ lấy phần development**. Không dùng phần test trong notebook này. `SEED = 42`. Áp dụng cách điền thiếu đã chọn ở notebook 03; chọn các cột số liên tục (bỏ `FLAG_*`).

## 1. IQR

**Lý thuyết — cần trình bày:**
- $IQR = Q_3 - Q_1$; ngoại lai nếu $x < Q_1 - 1.5\,IQR$ hoặc $x > Q_3 + 1.5\,IQR$.
- Đơn biến, không giả định phân phối chuẩn; nêu quy tắc gộp thành nhãn theo dòng (ví dụ ≥ 1 cột vượt ngưỡng).

> **[Code]** Tính ngưỡng trên train, đánh dấu theo từng cột và theo dòng; lưu tỷ lệ phát hiện.

> **[Nhận xét]** Cột nào bị đánh dấu nhiều nhất? Có hợp lý với phân phối lệch không?

## 2. Z-score

**Lý thuyết — cần trình bày:**
- $z = (x - \mu)/\sigma$, ngoại lai nếu $|z| > 3$.
- Giả định gần chuẩn; chính ngoại lai làm lệch $\mu$ và $\sigma$ (liên hệ kết quả kiểm định ở notebook 02).

> **[Code]** Đánh dấu theo cột và theo dòng; so sánh với IQR.

> **[Nhận xét]** Vì sao Z-score và IQR cho kết quả khác nhau trên biến lệch?

## 3. Isolation Forest (contamination ∈ {0.01, 0.05, 0.1})

**Lý thuyết — cần trình bày:**
- Ý tưởng: điểm bất thường bị cô lập sau ít lần chia ngẫu nhiên; $s(x) = 2^{-E[h(x)]/c(n)}$.
- `contamination` là tỷ lệ ngoại lai giả định — quyết định ngưỡng chứ không phải phát hiện ra tỷ lệ thật.

> **[Code]** `IsolationForest` cho 3 giá trị contamination, đa biến trên các cột đã chọn.

> **[Nhận xét]** Tập ngoại lai thay đổi thế nào khi tăng contamination?

## 4. LOF (n_neighbors ∈ {10, 20, 50})

**Lý thuyết — cần trình bày:**
- So sánh mật độ cục bộ của điểm với mật độ của láng giềng; LOF ≫ 1 là bất thường.
- Dựa trên khoảng cách → cần chuẩn hóa; tốn thời gian với n lớn.

> **[Code]** `LocalOutlierFactor` cho 3 giá trị n_neighbors trên dữ liệu đã chuẩn hóa (lấy mẫu nếu cần, ghi rõ).

> **[Nhận xét]** n_neighbors nhỏ/lớn phát hiện loại ngoại lai nào?

## 5. DBSCAN

**Lý thuyết — cần trình bày:**
- Điểm lõi, điểm biên, điểm nhiễu; tham số `eps`, `min_samples`.
- Chọn `eps` bằng k-distance plot; điểm nhiễu (nhãn −1) được xem là ngoại lai.

> **[Code]** Vẽ k-distance plot, chọn `eps`, chạy `DBSCAN` trên dữ liệu đã chuẩn hóa (lấy mẫu nếu cần).

> **[Nhận xét]** DBSCAN có phù hợp với dữ liệu nhiều chiều và mật độ không đều này không?

## 6. So sánh: tỷ lệ phát hiện và Jaccard

**Lý thuyết — cần trình bày:**
- $J(A, B) = \frac{|A \cap B|}{|A \cup B|}$ giữa tập ngoại lai của hai phương pháp.

> **[Code]** Bảng tỷ lệ phát hiện của mọi cấu hình (biến `outlier_rates`); ma trận Jaccard + heatmap.

> **[Nhận xét]** Phương pháp nào đồng thuận với nhau? Phương pháp đơn biến và đa biến khác nhau thế nào?

## 7. Tác động lên phân phối: kiểm định KS

**Lý thuyết — cần trình bày:**
- KS hai mẫu: $D = \sup_x |F_1(x) - F_2(x)|$; $H_0$: hai mẫu cùng phân phối.
- Lưu ý: tập trước và sau loại bỏ **chồng lên nhau** nên không độc lập → diễn giải thận trọng (có thể so sánh phần bị loại với phần giữ lại).

> **[Code]** Với mỗi phương pháp, chạy KS trên các cột chính trước/sau khi loại; lưu thống kê và p-value vào biến `ks_results`.

> **[Nhận xét]** Loại bỏ ngoại lai làm thay đổi phân phối nhiều không?

## 8. Quyết định xử lý

**Lý thuyết — cần trình bày:**
- Xóa dòng, cắt ngưỡng (winsorize/clip), biến đổi, hay giữ nguyên; giá trị cực đoan thật (ví dụ thu nhập rất cao) khác với lỗi.
- Không xóa dòng trong test; xử lý chỉ học tham số từ train.

> **[Code]** (Khuyến nghị) So sánh điểm CV của baseline khi giữ / xóa / cắt ngưỡng.

> **[Nhận xét]** Chọn cách xử lý và lý do.

## Tổng kết và lựa chọn

> **[Nhận xét]** Phương pháp và cách xử lý ngoại lai được dùng trong pipeline cuối.

## Hạn chế

> **[Nhận xét]** Kết quả trên mẫu, sự phụ thuộc vào tham số (eps, contamination), giới hạn của KS với mẫu chồng lấp.